In [1]:
import subprocess
import sys
import os
import time
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from tqdm import tqdm

Задаем начальные параметры генерации:

In [2]:
cpp_executable = "./gen"    # путь к скомпилированной C++ программе
output_file = "result.txt"  # файл, в который C++ программа пишет результат
reps = 1000;                # количество повторений метода Монте-Карло
n = 2000;                   # размер выборки
delta = 0.01;               # расстояние для постройки дистанционного графа
distrib = 2;                # вид распределения, 0 - uni, 1 - exp, 2 - norm 
click_size = 2;             # размер клики

input_args = list(map(str, [n, delta, reps, click_size, distrib]))
command = [cpp_executable] + input_args

Генерируем выборку:

In [7]:
res = list()
for i in range(1): 
    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE
    )
    with tqdm(total=100, unit="%") as pbar:
        while True:
            line = process.stdout.readline()
            if not line:
                break
            try:
                progress_str = line.split("Progress:")[0].strip()
                progress_value = float(progress_str)
                pbar.update(progress_value - pbar.n) # Update by the difference
            except ValueError:
                pass
    with open(output_file, 'r', encoding='utf-8') as f:
        result = f.read()
    result = list(map(float, result.split()))
    E = sum(result) / reps
    sq = sum(num * num for num in result)
    D = sq/reps - E*E
    res.append([E, D])

100%|█| 100.0/100 [00:00<00:00, 174.89


# Для 3-клик

In [8]:
match distrib:
    case 0: # Uniform
        P = 3 * delta**2
        P_ = 28/3 * delta**3
        P__ = 9 * delta**4 * 2
    case 1: # Exponential
        P = delta**2
        P_ = 7/3 * delta**3
        P__ = 9/5 * delta**4 * 2
    case 2: # Normal
        P = 0.276 * delta**2   #  получено с помощью Wolframa
        P_ = 0.283 * delta**3   # получено через подсчет ромбов на выборках
        P__ = 0.205 * delta**4  # получено перебором... :)

In [9]:
E = n**3/6*P
D = n**3/6*P + n**4/4*P_ + n**5/8*P__ - 9*n**5/36*P*P
print(round(E), round(D))

E = n*(n-1)*(n-2)/6 * P
D = n*(n-1)*(n-2)/6 * P + n*(n-1)*(n-2)*(n-3)/4 * P_ + n*(n-1)*(n-2)*(n-3)*(n-4)/8 * P__ \
    + n*(n-1)*(n-2)*(n-3)*(n-4)*(n-5)/36 * P * P - n*(n-1)*(n-2)*n*(n-1)*(n-2)/36 * P * P
print(round(E), round(D))

for i in res:
    print(round(E/i[0],3), round(D/i[1], 3))

36800 3274720
36745 3254684
3.258 62.928


# Для 2-клик

In [10]:
match distrib:
    case 0: # Uniform
        P = 2 * delta
        P_ = 8 * delta**2
    case 1: # Exponential
        P = delta
        P_ = 8/3 * delta**2
    case 2: # Normal
        P = 0.56419 * delta       # получено с помощью Wolframa
        P_ = 0.7351 * delta**2    # получено с помощью Wolframa

In [11]:
E = n**2/2*P
D = n**2/2*P + n**3/2*P_ - n**3 * P**2
print(round(E), round(D))

E = n*(n-1)/2 * P
D = n*(n-1)/2 * P + n*(n-1)*(n-2)/2 * P_ + n*(n-1)*(n-2)*(n-3)/4 * P * P - (n*(n-1)/2 * P)**2
print(round(E), round(D))

for i in res:
    print(round(E/i[0],3), round(D/i[1], 3))

11284 50676
11278 50547
1.0 0.977
